# 4.4.1.	Excedência dos padrões de qualidade do ar no Brasil

```{warning}
Não obtivemos o registro histório de todas as estações de monitoramento instaladas no Brasil. 
```

In [1]:
from IPython.display import HTML

HTML(r"""
<!DOCTYPE html>
<html lang="pt-br">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Mapa de Excedência - CONAMA 506/2024</title>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>

<style>
body {
  margin: 0;
  padding: 12px;
  font-family: Arial, sans-serif;
  background: #fff;
}
#controls {
  display: flex;
  flex-wrap: wrap;
  gap: 12px;
  align-items: center;
  margin-bottom: 10px;
}
.control {
  display: flex;
  align-items: center;
  gap: 8px;
  height: 38px;
}
.label {
  font-size: 14px;
  font-weight: 600;
  color: #333;
  white-space: nowrap;
}
.select {
  appearance: none;
  padding: 6px 10px;
  border: 1px solid #999;
  border-radius: 6px;
  background: #f9f9f9;
  cursor: pointer;
  font-size: 14px;
}
#status {
  font-size: 13px;
  color: crimson;
  margin-left: 8px;
}
#map {
  width: 100%;
  height: 640px;
  border: 1px solid #ddd;
  opacity: 0;
  transition: opacity .3s;
}
.leaflet-control.custom-legend {
  background: #fff;
  padding: 8px;
  border-radius: 6px;
  box-shadow: 0 1px 4px rgba(0,0,0,.4);
  font-size: 12px;
}
</style>
</head>

<body>
<h3>Mapa Interativo de Excedência de Padrões (CONAMA 506/2024)</h3>

<div id="controls">
  <div class="control">
    <span class="label">Padrão:</span>
    <select id="selPadrao" class="select">
      <option value="PI-1">PI-1</option>
      <option value="PI-2">PI-2</option>
      <option value="PI-3">PI-3</option>
      <option value="PI-4">PI-4</option>
      <option value="PF" selected>PF</option>
    </select>
  </div>

  <div class="control">
    <span class="label">Poluente:</span>
    <select id="selPol" class="select">
      <option value="MP25" selected>MP₂₅</option>
      <option value="MP10">MP₁₀</option>
      <option value="NO2">NO₂</option>
      <option value="SO2">SO₂</option>
      <option value="O3">O₃</option>
      <option value="CO">CO</option>
    </select>
  </div>

  <div class="control">
    <span class="label">Ano:</span>
    <select id="selAno" class="select"></select>
  </div>

  <div class="control" style="flex:1;">
    <span id="status"></span>
  </div>
</div>

<div id="map"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
// === CAMINHO BASE (ajuste conforme sua estrutura)
const BASE_PATH = "../_static/mapas/violacoes/";


// === Preenche anos dinamicamente ===
const selAno = document.getElementById("selAno");
for (let ano = 2024; ano >= 1998; ano--) {
  const opt = document.createElement("option");
  opt.value = ano;
  opt.textContent = ano;
  if (ano === 2024) opt.selected = true;
  selAno.appendChild(opt);
}

// === Poluentes formatados ===
const polFmt = {
  "MP10": "MP₁₀",
  "MP25": "MP₂₅",
  "NO2": "NO₂",
  "SO2": "SO₂",
  "O3": "O₃",
  "CO": "CO"
};

let map = null, layer = null, legend = null;

// === Inicialização do mapa ===
function ensureMap() {
  if (map) return;
  map = L.map('map', {
    minZoom: 3.8,
    maxZoom: 8,
    maxBounds: L.latLngBounds([-34, -74], [6, -34])
  }).setView([-14.2, -51.9], 4.3);

  L.tileLayer('https://cartodb-basemaps-a.global.ssl.fastly.net/light_all/{z}/{x}/{y}.png', {
    attribution: '© OpenStreetMap, © CartoDB'
  }).addTo(map);
  map.getContainer().style.opacity = 1;
}

// === Cores ===
function getColor(v) {
  v = Number(v);
  if (!isFinite(v)) return "gray";            // sem dado
  if (v === 0) return "rgb(0,180,0)";         // verde → 0 Excedência
  if (v <= 10) return "rgb(200,230,0)";       // amarelo-esverdeado
  if (v <= 20) return "rgb(255,220,0)";       // amarelo
  if (v <= 50) return "rgb(255,160,0)";       // laranja
  if (v <= 100) return "rgb(255,80,0)";       // laranja-escuro
  return "rgb(220,0,0)";                      // vermelho intenso
}

// === Legenda ===
function addLegend() {
  if (legend) map.removeControl(legend);
  legend = L.control({ position: "bottomleft" });
  legend.onAdd = function() {
    const div = L.DomUtil.create("div", "leaflet-control custom-legend");
    div.innerHTML =
      "<b>Faixas de Excedência:</b><br>" +
      "<div><span style='background:rgb(0,180,0);width:18px;height:10px;display:inline-block;'></span> 0</div>" +
      "<div><span style='background:rgb(200,230,0);width:18px;height:10px;display:inline-block;'></span> 1–10</div>" +
      "<div><span style='background:rgb(255,220,0);width:18px;height:10px;display:inline-block;'></span> 11–20</div>" +
      "<div><span style='background:rgb(255,160,0);width:18px;height:10px;display:inline-block;'></span> 21–50</div>" +
      "<div><span style='background:rgb(255,80,0);width:18px;height:10px;display:inline-block;'></span> 51–100</div>" +
      "<div><span style='background:rgb(220,0,0);width:18px;height:10px;display:inline-block;'></span> >100</div>";
    return div;
  };
  legend.addTo(map);
}


// === Popup ===
function popupHTML(p) {
  const viol = p.VIOLACOES || p.violacoes || "–";
  const nVal = p.N_VALIDOS || p.n_validos || "–";
  const exc = p.PCT_EXC || p.pct_exc;
  const excStr = (exc == null || isNaN(exc)) ? "inválido" : `${Number(exc).toFixed(1)}%`;
  const idMMA = p.ID_MMA_COMPLETO || p.id_mma_completo || "–";
  const idOema = p.ID_OEMA || p.id_oema || "–";
  const pol = polFmt[p.POLUENTE] || p.POLUENTE || "–";
  const pad = p.PADRAO || p.padrao || "–";
  const ano = p.ANO || p.ano || "–";

  return `
    <div style='font-family:Arial;font-size:12px;'>
      <b>${idMMA}</b><br>
      <i style='color:#555;'>Estação: ${idOema}</i><br>
      Poluente: ${pol}<br>
      Padrão: ${pad}<br>
      Ano: ${ano}<br>
      Dados válidos: ${nVal}<br>
      Excedência: ${viol}<br>
      Excedência: ${excStr}
    </div>`;
}

// === Atualização ===
async function updateMap() {
  ensureMap();
  const padrao = document.getElementById("selPadrao").value;
  const pol = document.getElementById("selPol").value;
  const ano = document.getElementById("selAno").value;
  const status = document.getElementById("status");

  status.textContent = `Carregando ${padrao}/${pol}/${ano}...`;
  const url = `${BASE_PATH}${padrao}/${pol}_${ano}.geojson`;

  try {
    const resp = await fetch(url);
    if (!resp.ok) throw new Error("404");
    const gj = await resp.json();
    if (layer) map.removeLayer(layer);
    if (legend) map.removeControl(legend);
    layer = L.geoJSON(gj, {
      pointToLayer: (f, latlng) => {
        const c = getColor(f.properties?.VIOLACOES);
        return L.circleMarker(latlng, {
          radius: 6,
          color: c,
          weight: 2,
          opacity: 0.4,
          fill: true,
          fillColor: c,
          fillOpacity: 0.55
        });
      },
      onEachFeature: (f, l) => l.bindPopup(popupHTML(f.properties || {}))
    }).addTo(map);
    addLegend();
    const b = layer.getBounds();
    if (b.isValid()) map.fitBounds(b, { padding: [20, 20] });
    status.textContent = "";
  } catch (e) {
    status.textContent = `⚠️ Dados indisponíveis para ${padrao}/${pol}/${ano}`;
    if (layer) map.removeLayer(layer);
    if (legend) map.removeControl(legend);
  }
}

// === Eventos ===
["selPadrao", "selPol", "selAno"].forEach(id => {
  document.getElementById(id).addEventListener("change", updateMap);
});

updateMap();
</script>
</body>
</html>
""")
